# 本notebook使用google colab运行

In [4]:
# Cell 1: 导入 TensorRT
import tensorrt as trt
import numpy as np

In [5]:
# Cell 2: 创建 builder 和 network
logger = trt.Logger(trt.Logger.VERBOSE)
builder = trt.Builder(logger)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, logger)

In [6]:
# Cell 3: 加载 ONNX 模型
with open("mnist_model.onnx", "rb") as f:
    if not parser.parse(f.read()):
        for error in range(parser.num_errors):
            print(parser.get_error(error))
        raise RuntimeError("Failed to parse ONNX model")

In [9]:
# Cell 4: 创建 builder config（启用 FP16）
config = builder.create_builder_config()
config.set_flag(trt.BuilderFlag.FP16)
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30) # 1GB

In [10]:
# IMPORTANT: Add optimization profile (required for explicit batch)
profile = builder.create_optimization_profile()
# Get input name from network
input_name = network.get_input(0).name
# Set min/opt/max shapes (all same for fixed size)
profile.set_shape(input_name, (1, 1, 28, 28), (1, 1, 28, 28), (1, 1, 28, 28))
config.add_optimization_profile(profile)

0

In [11]:
# Cell 5: 序列化 engine
serialized_engine = builder.build_serialized_network(network, config)
with open("mnist_model_fp16.engine", "wb") as f:
    f.write(serialized_engine)
print("TensorRT engine saved as mnist_model_fp16.engine")

TensorRT engine saved as mnist_model_fp16.engine


## 基础理解
### TensorRT 的 FP16 量化是什么意思？它如何加速推理？
- FP16是16bit的float，比默认的FP32位数更少，计算的时候运行速度会更快。
### trt.BuilderFlag.FP16 的作用是什么？如果模型是 INT8 量化，需要改什么？
- 让builder以FP16格式建造网络，加速后续计算。如果是INT8需要使用trt.BuilderFlag.INT8。
### 为什么需要“预热”（warm-up）后再测速度？不预热会怎样？
- 预热时会设置no_grad，不预热的话会让运行速度大幅下降。
- 补充：预热让 GPU 完成初始化，避免首次运行的开销
## 动手验证
### 把 mnist_model.onnx 转成 FP16 的 .engine 文件后，模型大小变化了多少？（用 os.path.getsize() 查看）
- .onxx为4689KB，.engine为2406KB，模型减少了一半左右。
### 对比 FP32 和 FP16 的推理结果，小数点后精度差多少？用 np.allclose() 验证
- 7位和4位？
- 需要验证。用 np.allclose() 跑一下就知道
### 如果你的 GPU 不支持 FP16（比如某些旧卡），运行 config.set_flag(trt.BuilderFlag.FP16) 会发生什么？
- 会报错。
## 深入思考
### 为什么小模型（如 MNIST）的 TensorRT 加速比可能不如大模型（如 ResNet50）？
- 因为gpu的初始化占用的资源和时间很可能远远大于一个小模型并行计算带来的时间减少。
### 如果部署到嵌入式设备（如 Jetson Nano），TensorRT 和 ONNX Runtime 哪个更合适？为什么？
- TensorRT更合适因为可以更好地利用gpu的并行计算能力，并且模型更小。